In [ ]:
import cv2

for i in range(3):
    c = cv2.VideoCapture(i)
    print(i, "열림" if c.isOpened() else "안 열림")
    c.release()

In [ ]:
import cv2, time
from pathlib import Path
from IPython.display import display, Image, clear_output

CLASS_NAME = "약"      # 지금 찍는 클래스: 약 / 기저귀 / 혈당측정키트 / 물티슈
N_SHOTS    = 100
INTERVAL   = 1.5        # 초. 이 시간 안에 물체 각도·위치 바꾸기
CAM_INDEX  = 1          # 셀 1에서 확인한 번호

save_dir = Path(f"dataset_raw/{CLASS_NAME}")
save_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"웹캠 인덱스 {CAM_INDEX} 못 엶. 0/2로도 시도해봐")

# 워밍업 (초기 프레임은 노출 자동조정 중이라 어두움)
for _ in range(10):
    cap.read()

try:
    for i in range(N_SHOTS):
        time.sleep(INTERVAL)
        ok, frame = cap.read()
        if not ok:
            print("프레임 못 읽음, 중단"); break

        path = save_dir / f"{CLASS_NAME}_{i:03d}.jpg"
        cv2.imwrite(str(path), frame)

        _, buf = cv2.imencode(".jpg", frame)
        clear_output(wait=True)
        display(Image(data=buf.tobytes()))
        print(f"[{i+1}/{N_SHOTS}] saved {path}")
finally:
    cap.release()

In [ ]:
import cv2, time
from IPython.display import display, Image, clear_output

CAM_INDEX = 0        # probe에서 실제 웹캠으로 확인된 번호
DURATION  = 15       # 초. 이 시간 동안 라이브 미리보기

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"인덱스 {CAM_INDEX} 못 엶")

# 워밍업 (초기 프레임은 노출 자동조정 중)
for _ in range(10):
    cap.read()

start = time.time()
try:
    while time.time() - start < DURATION:
        ok, frame = cap.read()
        if not ok:
            print("프레임 못 읽음"); break
        h, w = frame.shape[:2]
        _, buf = cv2.imencode(".jpg", frame)
        clear_output(wait=True)
        display(Image(data=buf.tobytes()))
        print(f"{w}x{h}  |  {int(time.time()-start)}s / {DURATION}s")
        time.sleep(0.1)   # 약 10fps
finally:
    cap.release()
    print("카메라 해제됨")